<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🔍</div><div style="color:white;font-size:1.5rem;font-weight:700;">Concordance Explorer</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Search for words or phrases and see them in context (KWIC)<br><a href="https://ladal.edu.au/tutorials/concordancing/concordancing.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.txt</code> files to <b>MyTexts</b> (Step 2). Demo texts are loaded automatically if the folder is empty.</li><li>Edit the <b>search term</b> and settings in the configuration cell (Step 2).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download results from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> Upload your files to the appropriate folder, edit the configuration cell, then click <b>Kernel &#x2192; Restart &amp; Run All</b>. The notebook runs from top to bottom automatically.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Step 1 — Load shared helpers (do not edit)
source("../helpers/ladal_helpers.R")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your texts &amp; configure</b></div>


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTexts</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left (click the folder icon if hidden).</li><li>Double-click the <b><code>MyTexts</code></b> folder.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b>.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted. If the folder is empty, demo data will be loaded automatically.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────
# Edit the values below, then run this cell (Shift+Enter).

search_term    <- "language"  # word or phrase to search for
window_size    <- 5           # words of context on each side (1–20)
case_sensitive <- FALSE       # TRUE = match exact capitalisation
use_regex      <- FALSE       # TRUE = treat search_term as a regular expression
max_display    <- 100         # max rows shown in the preview table


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
suppressPackageStartupMessages(library(quanteda))

# Seed demo data if MyTexts is empty
seed_demo_texts("MyTexts")

texts <- tryCatch(load_texts("MyTexts"),
  error = function(e) { .err(conditionMessage(e)); NULL })

if (!is.null(texts)) {
  show_corpus_summary(texts)
  .info(paste0("Searching for \"", search_term, "\"..."))
  toks   <- tokens(texts)
  vtype  <- if (use_regex) "regex" else "fixed"
  result <- kwic(toks, pattern = phrase(search_term),
                 window = window_size, valuetype = vtype,
                 case_insensitive = !case_sensitive)
  kwic_df <- as.data.frame(result)
  if (nrow(kwic_df) == 0) {
    .warn(paste0("No matches found for \"", search_term, "\". ",
                 "Try a different search term or check case_sensitive."))
  } else {
    # Per-document frequency summary
    freq_tbl <- as.data.frame(table(kwic_df$docname))
    names(freq_tbl) <- c("Document", "Hits")
    freq_rows <- apply(freq_tbl, 1, function(r) paste0(
      '<tr><td style="padding:3px 12px;font-size:.82rem;">', r["Document"],
      '</td><td style="padding:3px 12px;text-align:right;font-size:.82rem;">',
      r["Hits"], '</td></tr>'))
    IRdisplay::display_html(paste0(
      '<div style="margin:.5rem 0;font-family:sans-serif;">',
      '<b style="color:#51247a;font-size:.88rem;">Hits per document</b>',
      '<table style="border-collapse:collapse;margin-top:4px;">',
      '<thead><tr style="background:#f4f0f8;">',
      '<th style="padding:3px 12px;font-size:.8rem;color:#51247a;border-bottom:2px solid #d0c4e8;">Document</th>',
      '<th style="padding:3px 12px;font-size:.8rem;color:#51247a;border-bottom:2px solid #d0c4e8;">Hits</th>',
      '</tr></thead><tbody>', paste(freq_rows, collapse=""), '</tbody></table>',
      '<p style="font-size:.78rem;color:#888;margin-top:4px;">Total: ',
      nrow(kwic_df), ' match(es)</p></div>'
    ))
    # KWIC preview table
    display_df <- head(kwic_df[, c("docname","pre","keyword","post")], max_display)
    names(display_df) <- c("Document","Left context","Keyword","Right context")
    suppressPackageStartupMessages(library(htmltools))
    html_rows <- apply(display_df, 1, function(r) paste0(
      "<tr>",
      '<td style="color:#888;font-size:.8rem;white-space:nowrap;padding:3px 8px;">',
      htmltools::htmlEscape(r["Document"]), "</td>",
      '<td style="text-align:right;padding:3px 12px 3px 8px;color:#555;white-space:nowrap;">',
      htmltools::htmlEscape(r["Left context"]), "</td>",
      '<td style="font-weight:700;color:#51247a;padding:3px 8px;white-space:nowrap;">',
      htmltools::htmlEscape(r["Keyword"]), "</td>",
      '<td style="color:#555;padding:3px 8px;">',
      htmltools::htmlEscape(r["Right context"]), "</td></tr>"
    ))
    shown <- nrow(display_df); total <- nrow(kwic_df)
    caption <- if (shown < total)
      paste0("Showing first ", shown, " of ", total,
             " matches. Full results in concordance_results.xlsx.")
    else paste0("Showing all ", total, " match(es).")
    IRdisplay::display_html(paste0(
      '<div style="overflow-x:auto;max-height:500px;overflow-y:auto;">',
      '<table style="border-collapse:collapse;font-family:monospace;font-size:.85rem;width:100%;">',
      '<thead style="position:sticky;top:0;background:#f4f0f8;"><tr>',
      '<th style="text-align:left;padding:5px 8px;border-bottom:2px solid #51247a;color:#51247a;">Document</th>',
      '<th style="text-align:right;padding:5px 8px;border-bottom:2px solid #51247a;color:#51247a;">Left context</th>',
      '<th style="text-align:center;padding:5px 8px;border-bottom:2px solid #51247a;color:#51247a;">Keyword</th>',
      '<th style="text-align:left;padding:5px 8px;border-bottom:2px solid #51247a;color:#51247a;">Right context</th>',
      '</tr></thead><tbody>', paste(html_rows, collapse="\n"),
      '</tbody></table></div>',
      '<p style="font-size:.78rem;color:#888;margin-top:4px;">', caption, '</p>'
    ))
    save_excel(kwic_df, "concordance_results.xlsx")
    .ok("Saved <b>concordance_results.xlsx</b> to MyOutput.")
  }
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Files saved: <b>concordance_results.xlsx</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


## Citation

<div style="background:#faf7fd; border:1px solid #e0d4f0; border-left:5px solid #51247A; border-radius:6px; padding:16px 20px; margin:1.5rem 0; font-size:0.9rem; line-height:1.7; color:#333;">
  <div style="font-weight:700; color:#51247A; margin-bottom:8px;">&#x1F4CB; How to cite this tool</div>

Schweinberger, Martin. (2026). *LADAL Concordance Explorer* (Version 2.0.1). Brisbane: The University of Queensland. [https://ladal.edu.au/tools.html](https://ladal.edu.au/tools.html)

<details style="margin-top:12px;">
<summary style="cursor:pointer; font-size:0.82rem; color:#51247A; font-weight:600;">BibTeX</summary>

```bibtex
@software{schweinberger2026concordance,
  author       = {Schweinberger, Martin},
  title        = {LADAL Concordance Explorer},
  year         = {2026},
  version      = {2.0.1},
  organization = {The University of Queensland},
  url          = {https://ladal.edu.au/tools.html},
  doi          = {https://ladal.edu.au/tools.html}
}
```

</details>
</div>

In [ ]:
# Uncomment to show package versions for reproducibility
# sessionInfo()
